In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# v4 Cleaned Aug Smooth Weight Loss Inference

Self-contained inference notebook for the cleaned `v4` branch. It compares `best.pt` and `ema_best.pt` on validation, sweeps a global threshold on cached probabilities, then runs test inference and builds a submission zip.

In [ ]:

%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random, hashlib, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm
import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

cfg = {
    'run_dir': './runs/v4_cleaned_aug_smooth_weight_loss_infer',
    'ckpt_dir': './runs/v4_cleaned_aug_smooth_weight_loss',
    'img_hw': (384, 704),
    'val_batch_size': 1,
    'test_batch_size': 2,
    'val_num_workers': 0,
    'test_num_workers': 2,
    'seed': 42,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'use_amp': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_cuda = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('device:', device)
print('cuda devices:', num_cuda)
if num_cuda:
    for i in range(num_cuda):
        print(i, torch.cuda.get_device_name(i))

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2, ensure_ascii=False)


In [ ]:
from src.geometry import load_info_with_root, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

CAM_IDX = {name: i for i, name in enumerate(CAMERA_NAMES)}
LEFT_CAM = '/side/left/forward'
RIGHT_CAM = '/side/right/forward'
LEFT_IDX = CAM_IDX[LEFT_CAM]
RIGHT_IDX = CAM_IDX[RIGHT_CAM]

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [ ]:
def gaussian_kernel2d(kernel_size=5, sigma=0.8):
    ax = torch.arange(kernel_size, dtype=torch.float32) - kernel_size // 2
    xx, yy = torch.meshgrid(ax, ax, indexing='ij')
    kernel = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sigma * sigma))
    kernel = kernel / kernel.sum()
    return kernel


def blur_mask_np(mask_np: np.ndarray, kernel_t: torch.Tensor) -> np.ndarray:
    x = torch.from_numpy(mask_np.astype(np.float32))[None, None]
    k = kernel_t[None, None]
    pad = kernel_t.shape[-1] // 2
    y = F.conv2d(x, k, padding=pad)
    return y.squeeze().numpy()


def dilate_binary_np(mask_np: np.ndarray, width: int = 1) -> np.ndarray:
    if width <= 0:
        return mask_np.astype(bool)
    x = torch.from_numpy(mask_np.astype(np.float32))[None, None]
    k = torch.ones((1, 1, 2 * width + 1, 2 * width + 1), dtype=torch.float32)
    y = F.conv2d(x, k, padding=width)
    return (y.squeeze().numpy() > 0)


In [ ]:
class BEVDatasetV4CleanAugSmooth(Dataset):
    def __init__(self, info_df: pd.DataFrame, mode: str = 'train',
                 img_hw=(384, 704), aug: bool = False,
                 rover_vocab: dict | None = None,
                 post_scale_range=(0.90, 1.10),
                 rot_deg_range=(-3.0, 3.0),
                 flip_prob=0.5,
                 drop_images_prob=0.05,
                 max_dropped_images=2,
                 smooth_sigma=0.8,
                 smooth_kernel=5,
                 smooth_boundary_width=1):
        self.info = info_df.reset_index(drop=True).copy()
        self.mode = mode
        self.img_hw = img_hw
        self.aug = aug and (mode == 'train')
        self.rover_vocab = rover_vocab or {'__other__': 0}
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        self.post_scale_range = post_scale_range
        self.rot_deg_range = rot_deg_range
        self.flip_prob = flip_prob
        self.drop_images_prob = drop_images_prob
        self.max_dropped_images = max_dropped_images
        self.smooth_sigma = smooth_sigma
        self.smooth_kernel = smooth_kernel
        self.smooth_boundary_width = smooth_boundary_width
        self.gauss_kernel = gaussian_kernel2d(smooth_kernel, smooth_sigma)

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, row: pd.Series, key: str) -> Path:
        return resolve_info_path(Path(row['__data_root']), row[key])

    def _letterbox(self, img: Image.Image):
        src_W, src_H = img.size
        H_t, W_t = self.img_hw
        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))
        return canvas, s, pad_x, pad_y

    def _affine_params(self):
        if not self.aug:
            return 1.0, 0.0
        scale = random.uniform(*self.post_scale_range)
        rot_deg = random.uniform(*self.rot_deg_range)
        return scale, rot_deg

    def _apply_affine_canvas(self, canvas: Image.Image, scale: float, rot_deg: float):
        H_t, W_t = self.img_hw
        out = canvas
        if abs(scale - 1.0) > 1e-6:
            sW, sH = int(round(W_t * scale)), int(round(H_t * scale))
            scaled = out.resize((sW, sH), Image.BILINEAR)
            if scale >= 1.0:
                dx = random.randint(0, sW - W_t)
                dy = random.randint(0, sH - H_t)
                out = scaled.crop((dx, dy, dx + W_t, dy + H_t))
            else:
                out = Image.new('RGB', (W_t, H_t), (0, 0, 0))
                dx = (W_t - sW) // 2
                dy = (H_t - sH) // 2
                out.paste(scaled, (dx, dy))
        else:
            dx = 0
            dy = 0

        if abs(rot_deg) > 1e-6:
            out = out.rotate(rot_deg, resample=Image.BILINEAR, expand=False, fillcolor=(0, 0, 0))
        return out, dx, dy

    def _make_affine_matrix(self, scale: float, rot_deg: float, dx: float, dy: float):
        H_t, W_t = self.img_hw
        cx = (W_t - 1) / 2.0
        cy = (H_t - 1) / 2.0

        T1 = np.array([[1, 0, -cx], [0, 1, -cy], [0, 0, 1]], dtype=np.float32)
        S = np.array([[scale, 0, 0], [0, scale, 0], [0, 0, 1]], dtype=np.float32)
        T2 = np.array([[1, 0, cx - dx], [0, 1, cy - dy], [0, 0, 1]], dtype=np.float32)
        theta = np.deg2rad(rot_deg)
        c = np.cos(theta).astype(np.float32)
        s = np.sin(theta).astype(np.float32)
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
        return T2 @ R @ S @ T1

    def _load_one_camera(self, row: pd.Series, cam_key: str, intr_key: str, c2c_key: str,
                         scale: float, rot_deg: float, flip_lr: bool):
        img = Image.open(self._resolve_path(row, cam_key)).convert('RGB')
        intr_path = self._resolve_path(row, intr_key)
        car2cam_path = self._resolve_path(row, c2c_key)

        canvas, s0, pad_x, pad_y = self._letterbox(img)
        canvas, dx, dy = self._apply_affine_canvas(canvas, scale, rot_deg)

        if flip_lr:
            canvas = canvas.transpose(Image.FLIP_LEFT_RIGHT)

        arr = np.array(canvas)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s0; K[0, 2] *= s0
        K[1, 1] *= s0; K[1, 2] *= s0
        K[0, 2] += pad_x
        K[1, 2] += pad_y

        A = self._make_affine_matrix(scale, rot_deg, dx, dy)
        K = A @ K

        H_t, W_t = self.img_hw
        if flip_lr:
            Fm = np.array([[-1, 0, W_t - 1], [0, 1, 0], [0, 0, 1]], dtype=np.float32)
            K = Fm @ K

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def _swap_lr_if_needed(self, items, flip_lr: bool):
        if not flip_lr:
            return items
        items = list(items)
        items[LEFT_IDX], items[RIGHT_IDX] = items[RIGHT_IDX], items[LEFT_IDX]
        return items

    def _make_soft_targets(self, gt_hard: np.ndarray):
        valid = (gt_hard != 255)
        occ = (gt_hard == 1) & valid
        free = (gt_hard == 0) & valid

        occ_dil = dilate_binary_np(occ, self.smooth_boundary_width)
        free_dil = dilate_binary_np(free, self.smooth_boundary_width)
        boundary = (occ_dil & free_dil) & valid

        occ_blur = blur_mask_np(occ.astype(np.float32), self.gauss_kernel)
        occ_blur = np.clip(occ_blur, 0.0, 1.0)

        gt_soft = np.zeros_like(gt_hard, dtype=np.float32)
        gt_soft[free] = 0.0
        gt_soft[occ] = 1.0
        gt_soft[boundary] = occ_blur[boundary]
        gt_soft[~valid] = 0.0
        valid_mask = valid.astype(np.float32)
        return gt_soft, valid_mask

    def _drop_random_images(self, images: list[torch.Tensor]):
        if not self.aug:
            return images
        if random.random() >= self.drop_images_prob:
            return images
        n_drop = random.randint(1, min(self.max_dropped_images, len(images)))
        drop_ids = random.sample(range(len(images)), k=n_drop)
        images = list(images)
        for di in drop_ids:
            images[di] = torch.zeros_like(images[di])
        return images

    def _load_sample(self, idx: int):
        row = self.info.iloc[idx]
        scale, rot_deg = self._affine_params()
        flip_lr = self.aug and (random.random() < self.flip_prob)

        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row, cam, intr, c2c, scale=scale, rot_deg=rot_deg, flip_lr=flip_lr)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        imgs = self._swap_lr_if_needed(imgs, flip_lr)
        Ks = self._swap_lr_if_needed(Ks, flip_lr)
        M = self._swap_lr_if_needed(M, flip_lr)
        imgs = self._drop_random_images(imgs)

        out = {
            'images': torch.stack(imgs, dim=0),
            'intrinsics': torch.stack(Ks, dim=0),
            'car2cams': torch.stack(M, dim=0),
            'rover_id': torch.tensor(encode_rover(row.get('rover', '__other__'), self.rover_vocab), dtype=torch.long),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out

        gt_hard = np.load(self._resolve_path(row, GT_NAME)).squeeze()
        gt_hard = np.where(gt_hard < 0, 255, gt_hard).astype(np.int64)
        gt_soft, valid_mask = self._make_soft_targets(gt_hard)

        out['gt_hard'] = torch.from_numpy(gt_hard).unsqueeze(0)
        out['gt_soft'] = torch.from_numpy(gt_soft).unsqueeze(0)
        out['valid_mask'] = torch.from_numpy(valid_mask).unsqueeze(0)
        return out

    def __getitem__(self, idx: int):
        max_tries = 5
        last_err = None
        for k in range(max_tries):
            try_idx = (idx + k) % len(self.info)
            try:
                return self._load_sample(try_idx)
            except (OSError, ValueError, FileNotFoundError) as e:
                last_err = e
                continue
        raise RuntimeError(f'Failed to load sample after {max_tries} tries from idx={idx}: {last_err}')


In [ ]:
from src.models.decoder import SmallUNet, _UNetBlock

class _ResNet50Backbone(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        rn = torchvision.models.resnet50(weights=weights)
        self.stem = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.layer1 = rn.layer1
        self.layer2 = rn.layer2
        self.proj = nn.Conv2d(512, 128, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.proj(x)
        return x

class MultiCamBEVv4CleanAugSmooth(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _ResNet50Backbone(pretrained=False)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(128, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feat = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)


In [ ]:
from src.losses import _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch, streaming_threshold_sweep
from src.utils.training import unwrap_model, update_ema

class CompoundLossV2Soft(nn.Module):
    def __init__(self, pos_weight: float = 5.0,
                 weight_bce: float = 0.5,
                 weight_dice: float = 0.3,
                 weight_lovasz: float = 0.2):
        super().__init__()
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.weight_lovasz = weight_lovasz

    def forward(self, logits: torch.Tensor, gt_soft: torch.Tensor, gt_hard: torch.Tensor, valid_mask: torch.Tensor):
        valid = valid_mask > 0.5

        bce_per = F.binary_cross_entropy_with_logits(logits, gt_soft, pos_weight=self.pos_weight, reduction='none')
        bce = (bce_per * valid.float()).sum() / valid.float().sum().clamp(min=1.0)

        prob = torch.sigmoid(logits) * valid.float()
        gt_d = gt_soft * valid.float()
        inter = (prob * gt_d).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + gt_d.sum(dim=(1, 2, 3))
        dice = (1.0 - (2 * inter + 1) / (denom + 1)).mean()

        gt_h = (gt_hard == 1).float()
        lov_logits = logits[valid]
        lov_gt = gt_h[valid]
        lov = lovasz_hinge_flat(lov_logits, lov_gt) if lov_logits.numel() > 0 else logits.sum() * 0.0

        total = self.weight_bce * bce + self.weight_dice * dice + self.weight_lovasz * lov
        parts = {'bce': bce.item(), 'dice': dice.item(), 'lovasz': lov.item()}
        return total, parts

@torch.no_grad()
def evaluate_iou(model, loader, threshold=0.5):
    model.eval()
    inter, union = 0, 0
    for batch in loader:
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt_hard = batch['gt_hard'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        i, u = iou_binary_batch(logits, gt_hard, threshold=threshold)
        inter += i
        union += u
    return inter / max(union, 1)


In [ ]:


def find_ckpt_dir(preferred_dir: Path, fallback_dirs=None):
    fallback_dirs = fallback_dirs or []
    candidates = [preferred_dir] + [Path(p) for p in fallback_dirs]
    for cand in candidates:
        if (cand / 'best.pt').exists() or (cand / 'ema_best.pt').exists():
            return cand
    return preferred_dir


CKPT_DIR = find_ckpt_dir(
    Path(cfg['ckpt_dir']),
    fallback_dirs=[
        './runs/v4_cleaned_aug_smooth',
        './runs/v4_cleaned',
    ],
)
print('ckpt dir:', CKPT_DIR)
print('run dir :', RUN_DIR)


@torch.inference_mode()
def load_checkpoint(path: Path, device, model_kwargs: dict):
    ckpt = torch.load(path, map_location=device)
    rover_vocab = ckpt.get('rover_vocab', None)
    if rover_vocab is None:
        raise KeyError(f'rover_vocab not found in checkpoint: {path}')

    model = MultiCamBEVv4CleanAugSmooth(
        num_rover_classes=len(rover_vocab),
        rover_emb_dim=model_kwargs['rover_emb_dim'],
        rover_cond_dim=model_kwargs['rover_cond_dim'],
    ).to(device)

    state = ckpt['ema'] if 'ema' in ckpt else ckpt['model']
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f'loaded {path.name}: missing={len(missing)} unexpected={len(unexpected)}')
    if len(missing):
        print('  sample missing:', missing[:10])
    if len(unexpected):
        print('  sample unexpected:', unexpected[:10])
    model.eval()
    return model, ckpt, rover_vocab


@torch.inference_mode()
def collect_probs_and_gt(model, loader, cache_prefix: str, cache_dir: Path):
    cache_dir.mkdir(parents=True, exist_ok=True)
    probs_cache = cache_dir / f'{cache_prefix}_probs.pt'
    gt_cache = cache_dir / f'{cache_prefix}_gt.pt'

    if probs_cache.exists() and gt_cache.exists():
        print(f'loading cache: {probs_cache.name}')
        return torch.load(probs_cache, map_location='cpu'), torch.load(gt_cache, map_location='cpu')

    model.eval()
    probs_list = []
    gt_list = []

    for batch in tqdm(loader, desc=f'collect probs [{cache_prefix}]'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch.get('gt_hard', batch.get('gt')).cpu().to(torch.uint8)

        with torch.autocast('cuda', enabled=cfg['use_amp'] and device.type == 'cuda'):
            logits = model(images, intr, c2c, rover_id).float()
        probs_list.append(torch.sigmoid(logits).cpu().to(torch.float16))
        gt_list.append(gt)

    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0)
    torch.save(probs, probs_cache)
    torch.save(gt, gt_cache)
    return probs, gt


@torch.inference_mode()
def sweep_thresholds_from_probs(probs: torch.Tensor, gt_hard: torch.Tensor, thresholds):
    rows = []
    for t in thresholds:
        valid = gt_hard != 255
        gt_b = ((gt_hard == 1) & valid)
        pred = ((probs > t) & valid)
        inter = (pred & gt_b).sum().item()
        union = (pred | gt_b).sum().item()
        rows.append({'threshold': float(t), 'iou': float(inter / max(union, 1))})
    return pd.DataFrame(rows)


@torch.inference_mode()
def choose_best_threshold_from_probs(probs: torch.Tensor, gt_hard: torch.Tensor, thresholds):
    df = sweep_thresholds_from_probs(probs, gt_hard, thresholds)
    best_row = df.sort_values(['iou', 'threshold'], ascending=[False, False]).iloc[0]
    best_t = float(best_row['threshold'])
    best_iou = float(best_row['iou'])
    return best_t, best_iou, df


@torch.inference_mode()
def predict_test_probs(model, loader, cache_prefix: str, cache_dir: Path):
    cache_dir.mkdir(parents=True, exist_ok=True)
    probs_cache = cache_dir / f'{cache_prefix}_test_probs.pt'
    if probs_cache.exists():
        print(f'loading cache: {probs_cache.name}')
        return torch.load(probs_cache, map_location='cpu')

    model.eval()
    probs_list = []
    for batch in tqdm(loader, desc=f'test probs [{cache_prefix}]'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        with torch.autocast('cuda', enabled=cfg['use_amp'] and device.type == 'cuda'):
            logits = model(images, intr, c2c, rover_id).float()
        probs_list.append(torch.sigmoid(logits).cpu().to(torch.float16))
    probs = torch.cat(probs_list, dim=0)
    torch.save(probs, probs_cache)
    return probs


def select_prediction_name(row):
    if 'predicted_occupancy_grid' in row:
        return Path(row['predicted_occupancy_grid']).name
    if 'predicted_static_grid' in row:
        return Path(row['predicted_static_grid']).name
    return f'{int(row.name)}.npy'


def save_predictions(pred_probs: torch.Tensor, threshold: float, test_info: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    for p in out_dir.glob('*.npy'):
        p.unlink()
    preds = (pred_probs > threshold).numpy().astype(np.int32)
    assert len(preds) == len(test_info), (len(preds), len(test_info))
    for i, row in tqdm(test_info.iterrows(), total=len(test_info), desc=f'save npy t={threshold:.2f}'):
        name = select_prediction_name(row)
        grid = preds[i].reshape(1, BEV_H, BEV_W)
        np.save(out_dir / name, grid)


def build_submission_zip(test_info: pd.DataFrame, pred_dir: Path, sub_path: Path):
    if sub_path.exists():
        sub_path.unlink()
    with zipfile.ZipFile(sub_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(DATA_TEST / 'info.csv', arcname='info.csv')
        for npy in sorted(pred_dir.glob('*.npy')):
            zf.write(npy, arcname=f'predicted_static_grids/{npy.name}')
    with zipfile.ZipFile(sub_path, 'r') as zf:
        bad = zf.testzip()
        assert bad is None, f'Bad zip entry: {bad}'
        entries = zf.namelist()
    h = hashlib.sha256()
    with open(sub_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    print(json.dumps({
        'zip': str(sub_path.resolve()),
        'entries': len(entries),
        'sha256': h.hexdigest(),
        'size_mb': round(sub_path.stat().st_size / 1e6, 3),
    }, indent=2, ensure_ascii=False))


In [ ]:

clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()
print('train size:', len(train_info))
print('val size:', len(val_info))

reference_ckpt_path = None
for cand in [
    CKPT_DIR / 'best.pt',
    CKPT_DIR / 'ema_best.pt',
]:
    if cand.exists():
        reference_ckpt_path = cand
        break
if reference_ckpt_path is None:
    raise FileNotFoundError(f'No checkpoint found in {CKPT_DIR}')

reference_ckpt = torch.load(reference_ckpt_path, map_location='cpu')
if 'rover_vocab' in reference_ckpt:
    rover_vocab = reference_ckpt['rover_vocab']
    print('loaded rover_vocab from checkpoint')
else:
    rover_vocab, rover_stats = build_rover_vocab_from_train(
        train_info,
        min_count=cfg['min_rover_count'],
        topk=cfg['topk_rovers'],
    )
    print('built rover_vocab from train split')
    rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)

print('num rover classes including Other:', len(rover_vocab))
if 'rover_stats' not in locals():
    rover_stats = pd.DataFrame({'rover': list(rover_vocab.keys()), 'embedding_id': list(rover_vocab.values())})
    rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)

display(rover_stats.head(30))


In [ ]:

ds_val = BEVDatasetV4CleanAugSmooth(
    val_info,
    mode='val',
    img_hw=cfg['img_hw'],
    aug=False,
    rover_vocab=rover_vocab,
    post_scale_range=(0.90, 1.10),
    rot_deg_range=(-3.0, 3.0),
    flip_prob=0.5,
    drop_images_prob=0.05,
    max_dropped_images=2,
    smooth_sigma=0.8,
    smooth_kernel=5,
    smooth_boundary_width=1,
)
loader_val = DataLoader(
    ds_val,
    batch_size=cfg['val_batch_size'],
    shuffle=False,
    num_workers=cfg['val_num_workers'],
    pin_memory=(device.type == 'cuda'),
)

ds_test = BEVDatasetV4CleanAugSmooth(
    load_info_with_root(DATA_TEST, 'test'),
    mode='test',
    img_hw=cfg['img_hw'],
    aug=False,
    rover_vocab=rover_vocab,
    post_scale_range=(0.90, 1.10),
    rot_deg_range=(-3.0, 3.0),
    flip_prob=0.5,
    drop_images_prob=0.05,
    max_dropped_images=2,
    smooth_sigma=0.8,
    smooth_kernel=5,
    smooth_boundary_width=1,
)
loader_test = DataLoader(
    ds_test,
    batch_size=cfg['test_batch_size'],
    shuffle=False,
    num_workers=cfg['test_num_workers'],
    pin_memory=(device.type == 'cuda'),
)

debug_sample = ds_val[0]
for k, v in debug_sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)


In [ ]:

thresholds = [round(x, 2) for x in np.arange(0.20, 0.81, 0.02)]
ckpt_candidates = []
for name in ['best.pt', 'ema_best.pt']:
    p = CKPT_DIR / name
    if p.exists():
        ckpt_candidates.append(p)

if not ckpt_candidates:
    raise FileNotFoundError(f'No checkpoints found in {CKPT_DIR}')

results = []
for ckpt_path in ckpt_candidates:
    print('\n' + '=' * 100)
    print('checkpoint:', ckpt_path)
    model, ckpt, ckpt_vocab = load_checkpoint(
        ckpt_path,
        device,
        model_kwargs={'rover_emb_dim': cfg['rover_emb_dim'], 'rover_cond_dim': cfg['rover_cond_dim']},
    )

    if ckpt_vocab != rover_vocab:
        print('warning: checkpoint rover_vocab differs from loaded rover_vocab, using checkpoint vocab for this run')
        rover_vocab = ckpt_vocab
        ds_val.rover_vocab = rover_vocab
        ds_test.rover_vocab = rover_vocab

    val_probs, val_gt = collect_probs_and_gt(
        model,
        loader_val,
        cache_prefix=ckpt_path.stem,
        cache_dir=RUN_DIR / 'val_cache',
    )
    print('val_probs shape:', tuple(val_probs.shape))
    print('val_gt shape   :', tuple(val_gt.shape))

    best_t, best_iou, sweep_df = choose_best_threshold_from_probs(val_probs, val_gt, thresholds)
    sweep_path = RUN_DIR / f'{ckpt_path.stem}_val_sweep.csv'
    sweep_df.to_csv(sweep_path, index=False)
    print(f'best threshold for {ckpt_path.name}: {best_t:.2f} | IoU={best_iou:.6f}')

    results.append({
        'checkpoint': ckpt_path.name,
        'checkpoint_path': str(ckpt_path),
        'best_t': float(best_t),
        'best_iou': float(best_iou),
    })

results_df = pd.DataFrame(results).sort_values(['best_iou', 'checkpoint'], ascending=[False, True]).reset_index(drop=True)
display(results_df)

selected = results_df.iloc[0].to_dict()
selected_path = Path(selected['checkpoint_path'])
selected_threshold = float(selected['best_t'])
selected_name = selected_path.stem

# Prefer EMA if tied on IoU
if len(results_df) > 1:
    top_iou = results_df.iloc[0]['best_iou']
    tied = results_df[results_df['best_iou'] == top_iou].copy()
    if len(tied) > 1:
        ema_rows = tied[tied['checkpoint'].str.contains('ema', case=False, na=False)]
        if len(ema_rows):
            selected = ema_rows.iloc[0].to_dict()
            selected_path = Path(selected['checkpoint_path'])
            selected_threshold = float(selected['best_t'])
            selected_name = selected_path.stem

with open(RUN_DIR / 'selected_checkpoint.json', 'w') as f:
    json.dump({
        'selected_checkpoint': selected['checkpoint'],
        'selected_path': selected['checkpoint_path'],
        'best_t': selected_threshold,
        'best_iou': float(selected['best_iou']),
    }, f, indent=2, ensure_ascii=False)

print('\nselected checkpoint:', selected['checkpoint'])
print('selected threshold :', selected_threshold)
print('selected val iou   :', selected['best_iou'])

fig, ax = plt.subplots(figsize=(10, 4))
for ckpt_name in results_df['checkpoint'].tolist():
    df = pd.read_csv(RUN_DIR / f'{Path(ckpt_name).stem}_val_sweep.csv')
    ax.plot(df['threshold'], df['iou'], marker='o', label=ckpt_name)
ax.axvline(selected_threshold, color='red', linestyle='--', alpha=0.5, label=f'selected t={selected_threshold:.2f}')
ax.set_xlabel('threshold')
ax.set_ylabel('IoU')
ax.set_title('Validation threshold sweep')
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:

selected_ckpt_path = Path(selected['checkpoint_path'])
selected_model, selected_ckpt, _ = load_checkpoint(
    selected_ckpt_path,
    device,
    model_kwargs={'rover_emb_dim': cfg['rover_emb_dim'], 'rover_cond_dim': cfg['rover_cond_dim']},
)

print('using selected checkpoint for test:', selected_ckpt_path.name)

test_probs = predict_test_probs(
    selected_model,
    loader_test,
    cache_prefix=selected_ckpt_path.stem,
    cache_dir=RUN_DIR / 'test_cache',
)
print('test_probs shape:', tuple(test_probs.shape))

pred_dir = RUN_DIR / 'predicted_static_grids'
save_predictions(test_probs, selected_threshold, ds_test.info if hasattr(ds_test, 'info') else load_info_with_root(DATA_TEST, 'test'), pred_dir)

# sanity checks
saved = sorted(pred_dir.glob('*.npy'))
assert len(saved) == len(ds_test), (len(saved), len(ds_test))
for p in saved[:5]:
    arr = np.load(p)
    assert arr.shape == (1, BEV_H, BEV_W), arr.shape
    assert set(np.unique(arr).tolist()) <= {0, 1}, np.unique(arr)
print(f'predictions saved: {len(saved)} files')


In [ ]:

sub_path = RUN_DIR / f"submission_{RUN_DIR.name}_{selected_name}_t_{selected_threshold:.2f}.zip"
build_submission_zip(
    ds_test.info if hasattr(ds_test, 'info') else load_info_with_root(DATA_TEST, 'test'),
    RUN_DIR / 'predicted_static_grids',
    sub_path,
)
print('submission ready:', sub_path)
